# Opgave 1
## Spm. 1)
The branch and bound method is used to solve integer linear programming problems (ILP) by iteratively restricting the feasible region of a linear programming relaxation of the problem. The method involves branching on decision variables, creating subproblems, and using bounds to eliminate subproblems that cannot yield better solutions than the best known solution. The process continues until all subproblems have been explored or eliminated, resulting in the optimal integer solution.
## Spm. 2)
Since each branch from $P_1$ results in an integer solution in $P_2$ and $P_3$, and we cannot improve on a solution by further restricting the solution space, then we know:
1. That $P_2$ and $P_3$ will not have better integer solution.
2. It then follows that there are no further branches from $P_1$ that will be better, as $P_1$ it self is not a feasible integer solution.
## Spm. 3)
$P_4$ still has the possibility for a better solution as it is a feasible non integer solution and we still have variables that we can branch on. To check if a better solution exists on this branch we construct the LP in PULP. To do so i made a branch and bound solver.


In [1]:
import pulp as PLP
import numpy as np
def branch_and_bound(LPmodel, sense, best_so_far = [None], objectives = [None], problem = [0]):
    from math import floor, ceil
    import copy
    """
    :param LPmodel:
    input a Linear Programming relaxation of the ILP problem.
    Note it is import to keep track of if it is a minimization or maximization problem, as this will determine the
    branching strategy and pruning strategy.
    :return:
    Iterative Branch and Bound solution to the ILP problem.
    """
    if best_so_far[0] is None:
        if sense == "maximize":
            best_so_far[0] = -10 ** 6
        else:
            best_so_far[0] = 10 ** 6

    if sense != "maximize" and sense != "minimize":
        raise ValueError("sense must be either 'maximize' or 'minimize'")
    if sense == "maximize":
        def objective_is_not_better(obj):
            return obj < best_so_far[0]
    if sense == "minimize":
        def objective_is_not_better(obj):
            return obj > best_so_far[0]


    def is_integer_value():
        """
        :return:
        dict where keys are variable names and values are boolean values indicating whether variable is integer.
        """
        eps = 10**-4
        vars = LPmodel.variables()

        d = dict()
        # Assign boolean values to the decision variables based on whether they are integer or not
        for var in vars:
            if abs(var.varValue - ceil(var.varValue)) > eps and abs(var.varValue - floor(var.varValue)) > eps:
                d[var.name] = False
            else:
                d[var.name] = True
        return d
    print(20*"#")
    print("Problem : ", problem[0])
    print(20 * "#")
    print()
    ### Step 1 - Solve problem ###
    LPmodel.solve(PLP.PULP_CBC_CMD(msg = 0))
    obj = PLP.value(LPmodel.objective)
    ### Step 2 - Branch on non-integer decision variable
    vars = is_integer_value()
    if LPmodel.status == PLP.LpStatusInfeasible:
        print("Pruning branch, infeasible\n")
        return objectives
    if objective_is_not_better(obj):
        print("Pruning branch with objective value ", obj, " which is worse than best so far ", best_so_far[0])
        print("Decision variables: ")
        for v in LPmodel.variables():
            print(v.name, "=", v.varValue)
        print("Objective value: ", obj)
        print()
        return objectives
    if all([v for v in vars.values()]):
        print("Found feasible branch, backtracking")
        print("Decision variables: ")
        for v in LPmodel.variables():
            print(v.name, "=", v.varValue)
        print("Objective value: ", obj)
        print()
        if obj > best_so_far[0] and sense == "maximize":
            best_so_far[0] = obj
        if obj < best_so_far[0] and sense == "minimize":
            best_so_far[0] = obj
        objectives[0] = obj
        return objectives


    for name, value in vars.items():
        if value:
            continue
        else:
            branch_name = name
            branch_value = LPmodel.variablesDict()[branch_name].varValue
            break
    ### Step 3 - Create two branches and solve recursively ###
    ### Base case - All decision variables or the problem is not feasible or objective does not become better ###


    # Left branch
    left_model = copy.deepcopy(LPmodel)
    left_model += left_model.variablesDict()[branch_name] <= floor(branch_value)

    for v in LPmodel.variables():
        print(v.name, "=", v.varValue)
    print("Objective: ", obj)
    print("Adding constraint ", branch_name, " <= ", floor(branch_value), " to left branch")
    print()
    problem[0] = problem[0] + 1
    branch_and_bound(left_model, sense, best_so_far, objectives, problem)

    # Right branch
    right_model = copy.deepcopy(LPmodel)
    right_model += right_model.variablesDict()[branch_name] >= ceil(branch_value)
    print("Adding constraint ", branch_name, " >= ", ceil(branch_value), " to left branch")
    print()
    problem[0] = problem[0] + 1
    branch_and_bound(right_model, sense, best_so_far, objectives, problem)

    return best_so_far[0]


In [2]:
# LP problem
model = PLP.LpProblem("BranchAndBound", sense=PLP.LpMaximize)
# Define variables
var_range = range(5)
x = PLP.LpVariable.dicts("x", indices=var_range, cat=PLP.LpContinuous, lowBound=0, upBound=1)
# Objective
model += 6 * x[0] + 6 * x[1] + 4 * x[2] + 7 * x[3] + x[4], "Objective "
# Non-integer constraints
model += 3 * x[0] + 2 * x[1] + 5 * x[2] + 5 * x[3] + 6 * x[4] <= 14
model += 4 * x[0] + 5 * x[1] + 3 * x[2] + 7 * x[3] + 2 * x[4] <= 15
model += 3 * x[0] + 4 * x[1] + 7 * x[2] + 4 * x[3] + 1 * x[4] <= 10
branch_and_bound(model, sense = "maximize")


####################
Problem :  0
####################

x_0 = 1.0
x_1 = 0.75
x_2 = 0.0
x_3 = 1.0
x_4 = 0.0
Objective:  17.5
Adding constraint  x_1  <=  0  to left branch

####################
Problem :  1
####################

x_0 = 1.0
x_1 = 0.0
x_2 = 0.32432432
x_3 = 1.0
x_4 = 0.72972973
Objective:  15.027027010000001
Adding constraint  x_2  <=  0  to left branch

####################
Problem :  2
####################

Found feasible branch, backtracking
Decision variables: 
x_0 = 1.0
x_1 = 0.0
x_2 = 0.0
x_3 = 1.0
x_4 = 1.0
Objective value:  14.0

Adding constraint  x_2  >=  1  to left branch

####################
Problem :  3
####################

Pruning branch with objective value  10.0  which is worse than best so far  14.0
Decision variables: 
x_0 = 1.0
x_1 = 0.0
x_2 = 1.0
x_3 = 0.0
x_4 = 0.0
Objective value:  10.0

Adding constraint  x_1  >=  1  to left branch

####################
Problem :  4
####################

x_0 = 1.0
x_1 = 1.0
x_2 = 0.0
x_3 = 0.75
x_4 = 0.0
Objective: 

14.0

we see that there is more than one optimal solution, as $P_{10}$ above also yeilds 14 as the objective value like $P_2$ on the figure.